# 2. Neutralization and Perturbation

Refactored notebook version of domestic/international neutralization and publication perturbation scripts.


In [ ]:
from collections import defaultdict
from pathlib import Path
import gc, pickle, time
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
PICKLE_DIR = PROJECT_ROOT / 'data' / 'Openalex_2025' / 'pickles'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'results'
START_YEAR, END_YEAR = 2000, 2025
COUNTRIES = ['US','CN','GB','DE','JP','IT','FR','CA','IN','KR','ES','AU','BR','NL','TR']
COUNTRIES_OT = COUNTRIES + ['OT']
COUNTRIES_TOTAL = COUNTRIES + ['CN_Excluded','Total']
DOMAINS = ['Health Sciences','Life Sciences','Social Sciences','Physical Sciences']
FIELDS = ['Biochemistry, Genetics and Molecular Biology','Decision Sciences','Neuroscience','Veterinary','Immunology and Microbiology','Economics, Econometrics and Finance','Chemical Engineering','Medicine','Agricultural and Biological Sciences','Health Professions','Social sciences','Materials Science','Environmental Science','Earth and Planetary Sciences','Engineering','Dentistry','Physics and Astronomy','Business, Management and Accounting','Psychology','Energy','Arts and Humanities','Computer Science','Chemistry','Mathematics','Nursing','Pharmacology, Toxicology and Pharmaceutics']

def load_pickle(name):
    with open(PICKLE_DIR / name, 'rb') as fh:
        return pickle.load(fh)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def load_csr_arrays():
    mat = load_pickle('citation_matrix_csr.pickle')
    indptr, indices = mat.indptr, mat.indices
    del mat; gc.collect()
    return indptr, indices

def load_wos_valid_set():
    wos_issn = set(pd.read_csv(PICKLE_DIR / 'wos2025_Journal_ISSN_all.csv')['ISSN'])
    paper_issn = load_pickle('Paper_ISSN_list.pickle')
    valid = {pid for pid, issns in paper_issn.items() if any(x in wos_issn for x in issns)}
    valid.discard('')
    del wos_issn, paper_issn; gc.collect()
    return valid

def refs(indptr, indices, pid):
    return set(indices[indptr[pid]:indptr[pid+1]])

def norm_country(country, country_to_idx):
    return country if country in country_to_idx else 'OT'

def write_year_country(table, out_path, years, countries):
    ensure_dir(Path(out_path).parent)
    with open(out_path, 'w', encoding='utf-8') as fh:
        fh.write('year\t' + ''.join(f'{c}\t' for c in countries) + '\n')
        for year in years:
            fh.write(f'{year}\t' + ''.join(f'{table[year][c]}\t' for c in countries) + '\n')

def write_array(data, out_path, years, cols):
    ensure_dir(Path(out_path).parent)
    with open(out_path, 'w', encoding='utf-8') as fh:
        fh.write('year\t' + ''.join(f'{c}\t' for c in cols) + '\n')
        for i, year in enumerate(years):
            fh.write(f'{year}\t' + ''.join(f'{data[i,j]}\t' for j in range(len(cols))) + '\n')

def write_matrix_by_year(data, out_dir, prefix, years, countries):
    ensure_dir(out_dir)
    for year in years:
        with open(Path(out_dir)/f'{prefix}_{year}.txt', 'w', encoding='utf-8') as fh:
            fh.write('\\i(\\g(\\ab(d))\\-(i,j))\t' + ''.join(f'{c}\t' for c in countries) + '\n')
            for citing in countries:
                fh.write(f'{citing}\t')
                for cited in countries:
                    value = 0 if citing == cited else data[year][citing][cited]
                    fh.write(f'{value}\t')
                fh.write('\n')


## Citation-preference neutralization

In [ ]:
def convex_params(expected_df, raw_df, years, countries):
    lamb, center, expected = {}, {}, {}
    for year in years:
        lamb[year], center[year], expected[year] = {}, {}, {}
        for country in countries:
            f_exp, f_raw = float(expected_df.at[year,country]), float(raw_df.at[year,country])
            c = (1+f_exp)/2 if f_exp > f_raw else f_exp/2
            den = c - f_raw
            lamb[year][country] = 0.0 if abs(den) < 1e-12 else (f_exp-f_raw)/den
            center[year][country] = c
            expected[year][country] = f_exp
    return lamb, center, expected

def compute_domestic_neutralization():
    t0=time.time(); indptr,indices=load_csr_arrays(); valid=load_wos_valid_set()
    country=load_pickle('Paper_country_1st_aff.pickle'); year=load_pickle('Paper_year.pickle')
    years=list(range(START_YEAR,END_YEAR)); scenarios=['raw','convex_delta_reflength','convex_delta_otherref','center_delta_reflength','center_delta_otherref']
    cidx={c:i for i,c in enumerate(COUNTRIES_OT)}; tidx={c:i for i,c in enumerate(COUNTRIES)}; sidx={s:i for i,s in enumerate(scenarios)}
    exp_df=pd.read_csv(OUTPUT_ROOT/'p3_domestic_referencing_rate_delta/paper_global_share_10year_of_Country_in_Total.txt',sep='\t').set_index('year')
    raw_df=pd.read_csv(OUTPUT_ROOT/'p3_domestic_referencing_rate_delta/domestic_referencing_rate_mean_of_Country_in_Total.txt',sep='\t').set_index('year')
    lamb, center, expected = convex_params(exp_df, raw_df, years, COUNTRIES)
    acc=np.zeros((len(COUNTRIES),len(COUNTRIES_OT),len(years),len(scenarios)))
    for n,(pid,y) in enumerate(year.items(),1):
        if pid not in valid or not (START_YEAR <= y < END_YEAR): continue
        citing=country.get(pid)
        if citing is None: continue
        if n%100000==0: print('processed',n)
        citing=norm_country(citing,cidx); yi=y-START_YEAR
        counts=np.zeros(len(COUNTRIES_OT)); total=0.0
        for rid in refs(indptr,indices,pid):
            if rid not in valid: continue
            ry,rc=year.get(rid),country.get(rid)
            if ry is None or rc is None or not (y-10 <= ry < y): continue
            counts[cidx[norm_country(rc,cidx)]] += 1; total += 1
        vals=np.repeat(counts[:len(COUNTRIES),None],len(scenarios),axis=1)
        if citing in tidx and total>0:
            ti=tidx[citing]; domestic=counts[ti]; f_raw=domestic/total
            f_conv=(1-lamb[y][citing])*f_raw + lamb[y][citing]*center[y][citing]
            vals[ti,sidx['convex_delta_reflength']] += total*f_conv-domestic
            if abs(1-f_conv)>1e-12: vals[ti,sidx['convex_delta_otherref']] += (total*f_conv-domestic)/(1-f_conv)
            f_exp=expected[y][citing]
            vals[ti,sidx['center_delta_reflength']] += total*f_exp-domestic
            if abs(1-f_exp)>1e-12: vals[ti,sidx['center_delta_otherref']] += (total*f_exp-domestic)/(1-f_exp)
        acc[:,cidx[citing],yi,:] += vals
    out=OUTPUT_ROOT/'p3_domestic_referencing_rate_delta_normalize_convex'; ensure_dir(out)
    for target in COUNTRIES:
        ti=tidx[target]; den=acc[ti].sum(axis=0); num=den-acc[ti,ti]
        raw=num/np.maximum(den,1.0); base=np.where(raw[0]==0,1,raw[0])
        write_array(raw,out/f'ICR_of_Country_redelta_{target}.txt',years,scenarios)
        write_array(raw/base,out/f'ICR_norm_of_Country_redelta_{target}.txt',years,scenarios)
    print('time cost',time.time()-t0,'s')

def compute_international_neutralization():
    t0=time.time(); indptr,indices=load_csr_arrays(); valid=load_wos_valid_set()
    country=load_pickle('Paper_country_1st_aff.pickle'); year=load_pickle('Paper_year.pickle')
    years=range(START_YEAR,END_YEAR); scenarios=['raw','convex_delta_reflength','convex_delta_otherref','center_delta_reflength','center_delta_otherref']
    countries=['CN','IN','US','GB','DE','JP','IT','FR','CA','KR','ES','AU','BR','NL','TR','OT']; cidx={c:i for i,c in enumerate(countries)}
    pars={y:defaultdict(dict) for y in years}
    for y in years:
        exp=pd.read_csv(OUTPUT_ROOT/f'p4_inter_referencing_rate/inter_expected_referencing_rate_matrix_interval10_{y}.txt',sep='\t').set_index('\\i(\\g(\\ab(d))\\-(i,j))')
        raw=pd.read_csv(OUTPUT_ROOT/f'p4_inter_referencing_rate/inter_raw_referencing_rate_matrix_interval10_{y}.txt',sep='\t').set_index('\\i(\\g(\\ab(d))\\-(i,j))')
        for citing in countries:
            for cited in countries[:15]:
                f_exp,f_raw=exp[cited][citing],raw[cited][citing]; c=(1+f_exp)/2 if f_exp>f_raw else f_exp/2
                den=c-f_raw; lam=0 if abs(den)<1e-12 else (f_exp-f_raw)/den
                pars[y][citing][cited]=(f_exp,lam,c)
    out=OUTPUT_ROOT/'p4_inter_referencing_rate_delta_normalize_convex'; ensure_dir(out)
    for target in countries[:15]:
        print(target); totals={s:{y:[0,0] for y in years} for s in scenarios}; citing_count={c:{y:defaultdict(float) for y in years} for c in countries}
        for n,(pid,y) in enumerate(year.items(),1):
            if pid not in valid or not (START_YEAR <= y < END_YEAR) or pid not in country: continue
            citing=norm_country(country[pid],cidx); cnt=defaultdict(float)
            for rid in refs(indptr,indices,pid):
                if rid not in valid: continue
                ry,rc=year.get(rid),country.get(rid)
                if ry is None or rc is None or not (y-10 <= ry < y): continue
                cnt[norm_country(rc,cidx)] += 1; cnt['Total'] += 1
            for s in scenarios: cnt[s]=cnt[target]
            if citing != target:
                l_target=cnt[target]; l_total=cnt['Total']-cnt[citing]
                if l_total==0: continue
                f_raw=l_target/l_total; f_exp,lam,c=pars[y][citing][target]; f_conv=(1-lam)*f_raw+lam*c
                cnt['convex_delta_reflength'] += l_total*f_conv-l_target
                if abs(1-f_conv)>1e-12: cnt['convex_delta_otherref'] += (l_total*f_conv-l_target)/(1-f_conv)
                cnt['center_delta_reflength'] += l_total*f_exp-l_target
                if abs(1-f_exp)>1e-12: cnt['center_delta_otherref'] += (l_total*f_exp-l_target)/(1-f_exp)
            for s in scenarios: citing_count[citing][y][s] += cnt[s]
        raw_arr=np.zeros((END_YEAR-START_YEAR,len(scenarios)))
        for yi,y in enumerate(years):
            for si,s in enumerate(scenarios):
                for citing in countries:
                    if citing != target: totals[s][y][0] += citing_count[citing][y][s]
                    totals[s][y][1] += citing_count[citing][y][s]
                num,den=totals[s][y]; raw_arr[yi,si]=num/(den if den else 1)
        write_array(raw_arr,out/f'ICR_of_Country_redelta_{target}.txt',list(years),scenarios)
        write_array(raw_arr/np.where(raw_arr[0]==0,1,raw_arr[0]),out/f'ICR_norm_of_Country_redelta_{target}.txt',list(years),scenarios)
    print('time cost',time.time()-t0,'s')


In [ ]:
# compute_domestic_neutralization()
# compute_international_neutralization()


## Shared bootstrap engine

In [ ]:
from scipy.sparse import csr_matrix

class TargetBootstrap:
    def __init__(self,target_country,years,modify_entity='reference',n_bootstrap=1000,block_size=4,seed=20260422):
        self.target_country=target_country; self.years=np.asarray(list(years)); self.start=int(self.years[0]); self.end=int(self.years[-1]+1)
        self.modify_entity=modify_entity; self.n_bootstrap=n_bootstrap; self.block_size=block_size; self.seed=seed
        self.countries=COUNTRIES_OT; self.cidx={c:i for i,c in enumerate(self.countries)}; self.target_idx=self.cidx[target_country]; self.n_years=len(self.years)
    def _norm(self,c): return c if c in self.cidx else 'OT'
    def _groups(self,year_idx):
        return [(y,np.where(year_idx==y)[0].astype(np.int32)) for y in range(self.n_years) if np.where(year_idx==y)[0].size]
    def load(self):
        indptr,indices=load_csr_arrays(); valid=load_wos_valid_set(); country=load_pickle('Paper_country_1st_aff.pickle'); year=load_pickle('Paper_year.pickle')
        local={}; yidx=[]
        for pid,y in year.items():
            if pid in valid and country.get(pid)==self.target_country and self.start <= y < self.end:
                local[pid]=len(yidx); yidx.append(y-self.start)
        self.year_idx=np.asarray(yidx,dtype=np.int32); self.n_target=len(yidx); self.groups=self._groups(self.year_idx); print('Valid target papers:',self.n_target)
        nct=len(self.countries); baseline=np.zeros((nct,self.n_years,nct),dtype=np.int64); out_rows=[[] for _ in self.countries]; out_data=[[] for _ in self.countries]
        tt_rows=[]; tt_cols=[]; incoming_target=[defaultdict(int) for _ in range(self.n_years)]; incoming_foreign=[defaultdict(int) for _ in range(self.n_years)]
        for n,(pid,y) in enumerate(year.items(),1):
            if pid not in valid or pid not in country or not (self.start <= y < self.end): continue
            citing=self._norm(country[pid]); ci=self.cidx[citing]; yi=y-self.start; rset=refs(indptr,indices,pid)
            if not rset: continue
            is_target=pid in local
            if is_target: li=local[pid]; out=np.zeros(nct,dtype=np.int32); target_refs=[]
            for rid in rset:
                if rid not in valid: continue
                ry,rc=year.get(rid),country.get(rid)
                if ry is None or rc is None or not (y-10 <= ry < y): continue
                cited=self._norm(rc); cj=self.cidx[cited]; baseline[ci,yi,cj]+=1
                local_cited=local.get(rid) if cited==self.target_country else None
                if is_target:
                    out[cj]+=1
                    if local_cited is not None: target_refs.append(local_cited)
                if local_cited is not None:
                    (incoming_target if citing==self.target_country else incoming_foreign)[yi][local_cited]+=1
            if is_target:
                for cj,val in enumerate(out):
                    if val: out_rows[cj].append(li); out_data[cj].append(int(val))
                if target_refs: tt_rows.extend([li]*len(target_refs)); tt_cols.extend(target_refs)
        self.year_selector=csr_matrix((np.ones(self.n_target,dtype=np.int8),(self.year_idx,np.arange(self.n_target,dtype=np.int32))),shape=(self.n_years,self.n_target),dtype=np.int8)
        def pymat(rows,data):
            if not rows: return csr_matrix((self.n_target,self.n_years),dtype=np.int64)
            rows=np.asarray(rows,dtype=np.int32); return csr_matrix((np.asarray(data,dtype=np.int64),(rows,self.year_idx[rows])),shape=(self.n_target,self.n_years),dtype=np.int64)
        def incmat(inc):
            rows=[]; cols=[]; data=[]
            for yi,d in enumerate(inc): rows+=list(d.keys()); cols += [yi]*len(d); data += list(d.values())
            return csr_matrix((np.asarray(data,dtype=np.int64),(np.asarray(rows,dtype=np.int32),np.asarray(cols,dtype=np.int32))),shape=(self.n_target,self.n_years),dtype=np.int64) if rows else csr_matrix((self.n_target,self.n_years),dtype=np.int64)
        self.out=[pymat(out_rows[i],out_data[i]) for i in range(nct)]; self.in_target=incmat(incoming_target); self.in_foreign=incmat(incoming_foreign)
        self.target_target=csr_matrix((np.ones(len(tt_rows),dtype=np.int8),(np.asarray(tt_rows,dtype=np.int32),np.asarray(tt_cols,dtype=np.int32))),shape=(self.n_target,self.n_target),dtype=np.int8) if tt_rows else csr_matrix((self.n_target,self.n_target),dtype=np.int8)
        total=baseline.sum(axis=0); self.den=total.astype(float); self.num=np.zeros((self.n_years,nct))
        for j in range(nct): self.num[:,j]=total[:,j]-baseline[j,:,j]
    def draw_delete(self,rng,block_n,counts):
        mask=np.zeros((block_n,self.n_target),dtype=np.int32)
        for yi,idx in self.groups:
            k=int(counts[yi]);
            if k<=0: continue
            if k>=idx.size: mask[:,idx]=1
            else:
                for b in range(block_n): mask[b,rng.choice(idx,size=k,replace=False)]=1
        return mask.T
    def draw_add(self,rng,block_n,counts):
        arr=np.zeros((block_n,self.n_target),dtype=np.int32)
        for yi,idx in self.groups:
            k=int(counts[yi]);
            if k<=0: continue
            for b in range(block_n):
                chosen=rng.choice(idx,size=k,replace=True); u,c=np.unique(chosen,return_counts=True); arr[b,u]=c
        return arr.T
    def run(self,mode,counts):
        rng=np.random.default_rng(self.seed); raw=np.zeros((self.n_bootstrap,self.n_years,len(self.countries)))
        start=0
        while start<self.n_bootstrap:
            end=min(start+self.block_size,self.n_bootstrap); block=end-start; mat=self.draw_delete(rng,block,counts) if mode=='delete' else self.draw_add(rng,block,counts)
            num=np.zeros((block,self.n_years,len(self.countries))); den=np.zeros_like(num); sign=-1 if mode=='delete' else 1
            for j in range(len(self.countries)):
                if j==self.target_idx: continue
                out=self.out[j].T.dot(mat).T.astype(float,copy=False); num[:,:,j]=self.num[:,j]+sign*out; den[:,:,j]=self.den[:,j]+sign*out
            if self.modify_entity=='reference':
                out=self.out[self.target_idx].T.dot(mat).T.astype(float,copy=False); num[:,:,self.target_idx]=self.num[:,self.target_idx]; den[:,:,self.target_idx]=self.den[:,self.target_idx]+sign*out
            elif self.modify_entity=='paper':
                foreign=self.in_foreign.T.dot(mat).T.astype(float,copy=False); real=self.in_target.T.dot(mat).T.astype(float,copy=False); out=self.out[self.target_idx].T.dot(mat).T.astype(float,copy=False); inner=self.year_selector.dot(self.target_target.dot(mat)*mat).T.astype(float,copy=False)
                num[:,:,self.target_idx]=self.num[:,self.target_idx]+sign*foreign
                den[:,:,self.target_idx]=self.den[:,self.target_idx]+sign*foreign+sign*real+sign*out+(inner if mode=='add' else -sign*inner)
            raw[start:end]=np.divide(np.maximum(num,0) if mode=='delete' else num, np.maximum(np.maximum(den,0) if mode=='delete' else den,1.0)); start=end; print(mode,'finished',end,'/',self.n_bootstrap)
        base=np.where(raw[:,[0],:]==0,1,raw[:,[0],:]); norm=raw/base
        return raw.mean(0),np.percentile(raw,2.5,0),np.percentile(raw,97.5,0),norm.mean(0),np.percentile(norm,2.5,0),np.percentile(norm,97.5,0)

def write_bootstrap(result,out_dir,stem,years,countries=COUNTRIES_OT):
    raw,rl,rh,norm,nl,nh=result; ensure_dir(out_dir)
    write_array(raw,Path(out_dir)/f'ICR_of_Country_{stem}.txt',years,countries); write_array(norm,Path(out_dir)/f'ICR_norm_of_Country_{stem}.txt',years,countries)
    for arrs,prefix in [((rl,rh),'ICR_of_Country'),((nl,nh),'ICR_norm_of_Country')]:
        lo,hi=arrs
        with open(Path(out_dir)/f'{prefix}_{stem}_ci95.txt','w',encoding='utf-8') as fh:
            fh.write('year\t'+''.join(f'{c}_low\t{c}_high\t' for c in countries)+'\n')
            for i,y in enumerate(years): fh.write(f'{y}\t'+''.join(f'{lo[i,j]}\t{hi[i,j]}\t' for j in range(len(countries)))+'\n')


In [ ]:
# China count reduction
# boot = TargetBootstrap('CN', range(2000,2025), modify_entity='reference'); boot.load()
# for prob in [0.5,0.6,0.7,0.8,0.9]:
#     counts = np.zeros(boot.n_years, dtype=np.int32)
#     for yi,idx in boot.groups: counts[yi] = int(np.rint(prob * idx.size))
#     result = boot.run('delete', counts)
#     write_bootstrap(result, OUTPUT_ROOT / f'p5_paper_count_modify_bootstrap_simple/CN/modified_entity_{boot.modify_entity}', f'reduce_{prob}_count_CN', boot.years)

# India count addition
# boot = TargetBootstrap('IN', range(2000,2025), modify_entity='reference'); boot.load()
# for prob in [1.0,1.5,2.0,2.5,3.0,3.5,4.0]:
#     counts = np.zeros(boot.n_years, dtype=np.int32)
#     for yi,idx in boot.groups: counts[yi] = int(np.rint(prob * idx.size))
#     result = boot.run('add', counts)
#     write_bootstrap(result, OUTPUT_ROOT / f'p5_paper_count_modify_bootstrap_simple/IN/modified_entity_{boot.modify_entity}', f'add_{prob}_count_IN', boot.years)

In [ ]:
# China growth-rate reduction: c5_paper_rate_reduce_bootstrap_simple.py
# years = np.arange(2000, 2025)
# count_df = pd.read_csv(OUTPUT_ROOT/'p5_paper_count_modify_bootstrap_simple/Paper_Count_of_Country.txt', sep='\t').set_index('year')
# count_df['CN_excluded'] = count_df['Total'] - count_df['CN']
# delete_counts = np.zeros(len(years), dtype=np.int32)
# for i, year in enumerate(years):
#     expected_cn = count_df['CN'][2000] * count_df['CN_excluded'][year] / count_df['CN_excluded'][2000]
#     delete_counts[i] = int(np.rint(count_df['CN'][year] - expected_cn))
# boot = TargetBootstrap('CN', years, modify_entity='reference')
# boot.load()
# result = boot.run('delete', delete_counts)
# write_bootstrap(result, OUTPUT_ROOT/f'p5_paper_rate_modify_bootstrap_simple/CN/modified_entity_{boot.modify_entity}', 'reduce_rate_CN_excluded_2000', years)

# India growth-rate lifting: c5_paper_rate_add_bootstrap_simple_average.py
# years = np.arange(2000, 2025)
# count_df = pd.read_csv(OUTPUT_ROOT/'p5_paper_count_modify_bootstrap_simple/Paper_Count_of_Country.txt', sep='\t').query('2000 <= year < 2025').set_index('year')
# rate_df = pd.read_csv(OUTPUT_ROOT/'p5_paper_rate_modify_bootstrap_simple/Paper_Growth_Rate_of_Country.txt', sep='\t').query('2000 < year < 2025').set_index('year')
# rate_df['IN_CN_anchored_center'] = rate_df['CN'].mean()
# target_counts = [count_df['IN'][2000]]
# for year in years[1:]:
#     target_counts.append(int((rate_df['IN_CN_anchored_center'][year] + 1) * target_counts[-1]))
# add_counts = np.asarray(pd.Series(target_counts, index=years) - count_df['IN'], dtype=np.int32)
# boot = TargetBootstrap('IN', years, modify_entity='reference')
# boot.load()
# result = boot.run('add', add_counts)
# write_bootstrap(result, OUTPUT_ROOT/f'p5_paper_rate_modify_bootstrap_simple/IN/modified_entity_{boot.modify_entity}', 'add_rate_CN_anchored_center', years)